# 03 · Flow Matching 与 Action Chunk

这份 notebook 把“生成式策略”和“车端实时执行”放在同一个小实验里。它不是一个可直接上车的规划器，而是帮助你建立从扩散 / flow matching 到 VLA、World-Action Model（WA）以及 action chunk 的直觉。

学习目标：

- 用 NumPy 构造一个二维条件流，理解 source distribution、target distribution、velocity field 和 ODE rollout。
- 观察 Euler 步数、初始噪声和向量场近似误差怎样影响生成结果。
- 把“每次预测一段动作”放进一个简化的闭环控制器，分析 chunk horizon、re-plan period 和执行噪声。
- 记录 p50 / p95 推理耗时；真实智驾岗位通常同时关心模型质量、闭环稳定性和车端时延。

核心公式（使用 plain-text 记号，便于离线打开）：

- x_t = (1 - t) x_0 + t x_1
- v_target = x_1 - x_0
- dx / dt = v_theta(x, t)

其中 x_0 是容易采样的 source，x_1 是目标动作或轨迹分布。训练的核心是让 v_theta 在中间状态上逼近目标速度。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
from ipywidgets import interact, IntSlider, FloatSlider

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
rng = np.random.default_rng(7)


## Part A — 用一个小型流模型理解生成式轨迹

先做一个二维 toy problem：source 是起点附近的噪声，target 是两个“车道终点”组成的多模态分布。这个多模态设置对应真实驾驶中“跟车 / 换道 / 绕行”等多个可行动作模式。

为了让重点落在 flow matching 而不是框架工程上，我们用带 ridge 正则的线性特征回归近似 v_theta(x, t)。后面可以把同样的输入输出替换为 PyTorch MLP 或 Transformer。


In [ ]:
def sample_source(n, noise_scale=1.0, generator=rng):
    return generator.normal(loc=[0.0, 0.0], scale=[0.8, 0.8], size=(n, 2)) * noise_scale

def sample_target(n, lane_bias=0.0, generator=rng):
    mode = generator.integers(0, 2, size=n)
    lane = np.where(mode == 0, -1.5, 1.5)
    lane = lane + lane_bias + generator.normal(0.0, 0.25, size=n)
    longitudinal = 10.0 + generator.normal(0.0, 0.4, size=n)
    return np.c_[longitudinal, lane]

n_pairs = 6000
x0 = sample_source(n_pairs)
x1 = sample_target(n_pairs)
t = rng.uniform(0.0, 1.0, size=n_pairs)
xt = (1.0 - t[:, None]) * x0 + t[:, None] * x1
v_target = x1 - x0

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(x0[:, 0], x0[:, 1], s=4, alpha=0.25)
ax[0].set_title('source x0')
ax[0].set_xlabel('longitudinal')
ax[0].set_ylabel('lateral')
ax[1].scatter(x1[:, 0], x1[:, 1], s=4, alpha=0.25, c=np.where(x1[:, 1] > 0, 'tab:orange', 'tab:blue'))
ax[1].set_title('target x1: two lane modes')
ax[1].set_xlabel('longitudinal')
plt.tight_layout()


In [ ]:
def features(x, t_value):
    x = np.atleast_2d(np.asarray(x, dtype=float))
    t_value = np.asarray(t_value, dtype=float).reshape(-1, 1)
    if len(t_value) == 1 and len(x) > 1:
        t_value = np.repeat(t_value, len(x), axis=0)
    return np.c_[
        np.ones(len(x)),
        x[:, 0],
        x[:, 1],
        t_value[:, 0],
        x[:, 0] * t_value[:, 0],
        x[:, 1] * t_value[:, 0],
        t_value[:, 0] ** 2,
    ]

ridge = 1e-3
phi = features(xt, t)
weights = np.linalg.solve(
    phi.T @ phi + ridge * np.eye(phi.shape[1]),
    phi.T @ v_target,
)

def predict_velocity(x, t_value):
    return features(x, t_value) @ weights

training_rmse = np.sqrt(np.mean((predict_velocity(xt, t) - v_target) ** 2))
print(f'flow-field regression RMSE: {training_rmse:.4f}')


In [ ]:
def rollout(n=300, steps=32, noise_scale=1.0, seed=9):
    local_rng = np.random.default_rng(seed)
    x = sample_source(n, noise_scale=noise_scale, generator=local_rng)
    for step in range(steps):
        t_value = (step + 0.5) / steps
        x = x + predict_velocity(x, t_value) / steps
    return x

generated = rollout()
target_for_plot = sample_target(len(generated), generator=np.random.default_rng(11))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(target_for_plot[:, 0], target_for_plot[:, 1], s=7, alpha=0.20, label='target samples')
ax.scatter(generated[:, 0], generated[:, 1], s=10, alpha=0.55, label='Euler rollout')
ax.set_title('生成结果：向目标动作分布推进')
ax.set_xlabel('longitudinal action / endpoint')
ax.set_ylabel('lateral action / endpoint')
ax.legend()
plt.show()


### 交互练习 1：步数和初始噪声

调节 steps，观察离散 ODE 的积分误差；调节 noise_scale，观察 source 偏离训练分布后是否会产生异常终点。

建议记录：

1. 在 steps = 4, 8, 16, 32, 64 下，终点到最近 target mode 中心的平均距离。
2. noise_scale 增大后，哪一个方向的误差先恶化？
3. 把 features 增加二次项或交叉项，重新比较 regression RMSE 和 rollout 质量。


In [ ]:
def show_flow(steps=32, noise_scale=1.0):
    generated = rollout(n=300, steps=steps, noise_scale=noise_scale, seed=9)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(x1[:, 0], x1[:, 1], s=4, alpha=0.12, label='training targets')
    ax.scatter(generated[:, 0], generated[:, 1], s=10, alpha=0.60, label='rollout')
    ax.set_xlim(-3, 13)
    ax.set_ylim(-5, 5)
    ax.set_title(f'steps={steps}, noise_scale={noise_scale:.1f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend()
    plt.show()

interact(
    show_flow,
    steps=IntSlider(min=2, max=96, step=2, value=32, description='Euler steps'),
    noise_scale=FloatSlider(min=0.25, max=2.5, step=0.25, value=1.0, description='source noise'),
);


## Part B — Action chunk：模型预测一段，系统分段执行

VLA / WA 类策略常常不只输出一个瞬时动作，而是输出一个 action chunk。这样能减少每步调用大模型的频率，但也引入两个系统问题：

- chunk_horizon 太短：调用频繁，算力和通信压力上升。
- policy_period 太长：观测变旧，动作可能已经不适合当前场景。
- perception latency、执行噪声和突发障碍会放大 stale action 风险。

下面的 toy controller 每次根据当前状态生成未来若干个二维速度动作，只执行其中一段，然后重新规划。二维位置只是为了可视化；真实系统中动作可以是轨迹点、曲率、加速度或 control command。


In [ ]:
dt = 0.1
horizon = 80
time_grid = np.arange(horizon) * dt
lane_change_start = 2.0
lane_change_end = 5.5
progress = np.clip((time_grid - lane_change_start) / (lane_change_end - lane_change_start), 0.0, 1.0)
reference = np.c_[8.0 * time_grid, 1.8 * (3 * progress ** 2 - 2 * progress ** 3)]

def simulate_chunked_control(chunk_horizon=8, policy_period=4, action_noise=0.0):
    local_rng = np.random.default_rng(21)
    position = reference[0].copy()
    trajectory = [position.copy()]
    commands = []
    current_chunk = np.zeros((chunk_horizon, 2))
    for k in range(horizon - 1):
        if k % policy_period == 0:
            future_indices = np.minimum(
                k + np.arange(1, chunk_horizon + 1), horizon - 1
            )
            current_chunk = (reference[future_indices] - position) / dt
            current_chunk += local_rng.normal(0.0, action_noise, size=current_chunk.shape)
        chunk_index = min(k % policy_period, len(current_chunk) - 1)
        command = current_chunk[chunk_index]
        position = position + command * dt
        trajectory.append(position.copy())
        commands.append(command.copy())
    return np.asarray(trajectory), np.asarray(commands)

def show_chunk_control(chunk_horizon=8, policy_period=4, action_noise=0.0):
    trajectory, commands = simulate_chunked_control(
        chunk_horizon=chunk_horizon,
        policy_period=policy_period,
        action_noise=action_noise,
    )
    rmse = np.sqrt(np.mean((trajectory - reference) ** 2))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(reference[:, 0], reference[:, 1], label='reference', linewidth=2)
    ax[0].plot(trajectory[:, 0], trajectory[:, 1], label='executed', linewidth=2)
    ax[0].scatter(reference[0, 0], reference[0, 1], marker='o', label='start')
    ax[0].set_title(f'closed-loop path, RMSE={rmse:.3f}')
    ax[0].set_xlabel('x / m')
    ax[0].set_ylabel('y / m')
    ax[0].legend()
    ax[1].plot(time_grid[:-1], commands[:, 0], label='longitudinal command')
    ax[1].plot(time_grid[:-1], commands[:, 1], label='lateral command')
    ax[1].set_title('executed action stream')
    ax[1].set_xlabel('time / s')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(
    show_chunk_control,
    chunk_horizon=IntSlider(min=1, max=20, step=1, value=8, description='chunk H'),
    policy_period=IntSlider(min=1, max=20, step=1, value=4, description='replan K'),
    action_noise=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.0, description='noise'),
);


### 交互练习 2：把“模型能力”翻译成“系统指标”

请完成下面的实验记录：

- 固定 policy_period=4，比较 chunk_horizon=2/8/16 的闭环 RMSE。
- 固定 chunk_horizon=8，增大 policy_period，找到 lateral error 明显变大的拐点。
- 在噪声存在时加入一个简单 jerk / lateral-velocity clip，再比较轨迹是否更平滑。
- 把 action_noise 视为模型不确定性，设计一个 fallback 条件：不确定性高时缩短执行段或降低速度。

这里的关键不是让 toy controller 看起来“聪明”，而是练习把模型输出形式、调用频率、闭环误差和安全余量放进同一个实验。


In [ ]:
def benchmark_velocity_field(repeats=1000):
    query = np.zeros((1, 2))
    latencies_us = []
    for _ in range(repeats):
        start = perf_counter()
        _ = predict_velocity(query, 0.5)
        latencies_us.append((perf_counter() - start) * 1e6)
    latencies_us = np.asarray(latencies_us)
    print(f'predict_velocity p50: {np.percentile(latencies_us, 50):.2f} us')
    print(f'predict_velocity p95: {np.percentile(latencies_us, 95):.2f} us')
    print(f'predict_velocity p99: {np.percentile(latencies_us, 99):.2f} us')

benchmark_velocity_field()


## Optional：用一个 PyTorch MLP 替换线性向量场

如果环境里已经安装了 PyTorch，运行下面的 cell 可以把上面的线性回归替换成一个小 MLP。它不是为了追求 benchmark，而是帮助你把 toy flow matching 映射到真实训练代码：输入是状态和时间，监督信号是目标 velocity。

如果没有 PyTorch，cell 会安全跳过；可以先完成 NumPy 练习，再按自己的 CUDA 环境安装。


In [ ]:
try:
    import torch
    import torch.nn as nn

    torch.set_num_threads(1)
    torch.manual_seed(7)
    train_x = torch.tensor(np.c_[xt, t], dtype=torch.float32)
    train_y = torch.tensor(v_target, dtype=torch.float32)
    model = nn.Sequential(
        nn.Linear(3, 64),
        nn.SiLU(),
        nn.Linear(64, 64),
        nn.SiLU(),
        nn.Linear(64, 2),
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
    for step in range(300):
        batch = torch.randint(0, len(train_x), (256,))
        prediction = model(train_x[batch])
        loss = ((prediction - train_y[batch]) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        final_loss = ((model(train_x) - train_y) ** 2).mean().item()
    print(f'PyTorch MLP training MSE: {final_loss:.5f}')
except Exception as exc:
    print('PyTorch optional cell skipped:', type(exc).__name__)
    print('Install PyTorch separately for your platform if you want to run it.')


## 完成标准

完成本 notebook 后，你应能在项目 README 或实验记录中回答：

1. flow matching 的 source、target、interpolation path、velocity target 分别是什么？
2. Euler step 数、source noise 和 vector-field approximation 如何影响终点分布？
3. action chunk 的 horizon、re-plan period、观测延迟和不确定性之间有什么 trade-off？
4. 至少给出一张参数对比图、一项失败案例和 p50/p95 时延记录。
5. 下一步如何把二维 toy action 换成 nuPlan / NAVSIM 中的轨迹或控制量？
